# DermaScope - Notebook Pengolahan Citra

Notebook ini dipakai untuk uji awal dan bahan gambar laporan. Alurnya dibuat ringkas: unggah foto, deteksi wajah, siapkan area wajah, baca sinyal visual kulit, buat overlay, lalu tampilkan tabel hasil.

Nilai dihitung dari piksel gambar. Notebook ini memetakan sinyal visual pada foto, bukan diagnosis medis.


## 1. Instalasi Pustaka

OpenCV dipakai untuk membaca dan mengolah gambar. NumPy untuk operasi piksel. Pandas untuk tabel. Matplotlib untuk menampilkan gambar. scikit-image dipakai untuk fitur tekstur GLCM.


In [ ]:
!pip -q install opencv-python-headless numpy pandas matplotlib scikit-image


## 2. Import dan Konfigurasi Awal


In [ ]:
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from skimage.feature import graycomatrix, graycoprops

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

INFO_SINYAL = {
    "jerawat": {"label": "Jerawat", "singkat": "Jrwt", "warna": (31, 90, 255), "bobot_luas": 2.2, "bobot_jumlah": 1.8},
    "noda_gelap": {"label": "Noda gelap", "singkat": "Noda", "warna": (0, 77, 138), "bobot_luas": 2.0, "bobot_jumlah": 0.8},
    "kerutan": {"label": "Kerutan", "singkat": "Garis", "warna": (143, 123, 0), "bobot_luas": 1.7, "bobot_jumlah": 0.15},
    "kemerahan": {"label": "Kemerahan", "singkat": "Merah", "warna": (127, 0, 209), "bobot_luas": 1.6, "bobot_jumlah": 0.25},
    "pori_besar": {"label": "Pori besar", "singkat": "Pori", "warna": (31, 122, 97), "bobot_luas": 1.4, "bobot_jumlah": 0.08},
}

URUTAN_SINYAL = [isi["label"] for isi in INFO_SINYAL.values()]


## 3. Fungsi Menyiapkan Wajah

Kode ini bisa dipakai sebagai **Gambar 1.1 Kode Menyiapkan Wajah**. Prosesnya: deteksi wajah, perluasan area wajah, pemotongan gambar, lalu perataan cahaya.


In [ ]:
def deteksi_wajah(gambar):
    abu_abu = cv.cvtColor(gambar, cv.COLOR_BGR2GRAY)
    berkas_model = cv.data.haarcascades + "haarcascade_frontalface_default.xml"
    pencari_wajah = cv.CascadeClassifier(berkas_model)
    daftar_wajah = pencari_wajah.detectMultiScale(abu_abu, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    if len(daftar_wajah) > 0:
        x, y, lebar, tinggi = max(daftar_wajah, key=lambda area: area[2] * area[3])
        return (int(x), int(y), int(lebar), int(tinggi)), True

    tinggi_gambar, lebar_gambar = gambar.shape[:2]
    ukuran = int(min(lebar_gambar, tinggi_gambar) * 0.72)
    x = max(0, (lebar_gambar - ukuran) // 2)
    y = max(0, int((tinggi_gambar - ukuran) * 0.42))
    return (x, y, min(ukuran, lebar_gambar - x), min(ukuran, tinggi_gambar - y)), False


def perluas_area_wajah(area_wajah, lebar_gambar, tinggi_gambar):
    x, y, lebar, tinggi = area_wajah
    tambah_samping = int(lebar * 0.26)
    tambah_atas = int(tinggi * 0.30)
    tambah_bawah = int(tinggi * 0.18)

    kiri = max(0, x - tambah_samping)
    atas = max(0, y - tambah_atas)
    kanan = min(lebar_gambar, x + lebar + tambah_samping)
    bawah = min(tinggi_gambar, y + tinggi + tambah_bawah)
    return kiri, atas, max(1, kanan - kiri), max(1, bawah - atas)


def ratakan_cahaya(area_wajah):
    lab = cv.cvtColor(area_wajah, cv.COLOR_BGR2LAB)
    cahaya, warna_a, warna_b = cv.split(lab)
    perata = cv.createCLAHE(clipLimit=1.8, tileGridSize=(8, 8))
    cahaya_rata = perata.apply(cahaya)
    return cv.cvtColor(cv.merge((cahaya_rata, warna_a, warna_b)), cv.COLOR_LAB2BGR)


def siapkan_wajah(gambar):
    area_wajah, wajah_ditemukan = deteksi_wajah(gambar)
    x, y, lebar, tinggi = perluas_area_wajah(area_wajah, gambar.shape[1], gambar.shape[0])
    potongan_wajah = gambar[y:y + tinggi, x:x + lebar]
    return ratakan_cahaya(potongan_wajah), area_wajah, (x, y, lebar, tinggi), wajah_ditemukan


## 4. Fungsi Membaca Sinyal Visual Kulit

Kode ini bisa dipakai sebagai **Gambar 1.2 Kode Membaca Sinyal Visual Kulit**. Sinyal dihitung dari mask kulit, zona wajah, warna, garis halus, titik kecil, dan tekstur GLCM.


In [ ]:
def jadi_mask(syarat):
    return np.where(syarat, 255, 0).astype(np.uint8)


def buat_mask_kulit(wajah):
    warna_ycrcb = cv.cvtColor(wajah, cv.COLOR_BGR2YCrCb)
    warna_hsv = cv.cvtColor(wajah, cv.COLOR_BGR2HSV)

    mask_warna = cv.inRange(warna_ycrcb, np.array([30, 128, 75]), np.array([245, 188, 148]))
    mask_hsv = cv.inRange(warna_hsv, np.array([0, 18, 45]), np.array([35, 185, 255]))
    mask = cv.bitwise_and(mask_warna, mask_hsv)

    kotak_5 = np.ones((5, 5), dtype=np.uint8)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN, kotak_5)
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, kotak_5)

    if np.count_nonzero(mask) < wajah.shape[0] * wajah.shape[1] * 0.12:
        tinggi, lebar = wajah.shape[:2]
        mask = np.zeros(wajah.shape[:2], dtype=np.uint8)
        pusat = (lebar // 2, int(tinggi * 0.54))
        ukuran = (int(lebar * 0.36), int(tinggi * 0.42))
        cv.ellipse(mask, pusat, ukuran, 0, 0, 360, 255, -1)

    return mask


ZONA_WAJAH = (
    ("dahi", "Dahi", 0.20, 0.08, 0.60, 0.20),
    ("pipi_kiri", "Pipi kiri", 0.00, 0.32, 0.42, 0.38),
    ("pipi_kanan", "Pipi kanan", 0.58, 0.32, 0.42, 0.38),
    ("hidung", "Hidung", 0.38, 0.30, 0.24, 0.44),
    ("dagu", "Dagu", 0.26, 0.70, 0.48, 0.30),
)


def bagi_zona_wajah(lebar, tinggi):
    return [
        (kode, nama, (int(lebar * x), int(tinggi * y), int(lebar * w), int(tinggi * h)))
        for kode, nama, x, y, w, h in ZONA_WAJAH
    ]


def saring_area_sinyal(mask, area_min, area_maks):
    jumlah, label, data_area, _ = cv.connectedComponentsWithStats(mask, connectivity=8)
    hasil = np.zeros_like(mask)

    for nomor in range(1, jumlah):
        luas = int(data_area[nomor, cv.CC_STAT_AREA])
        if area_min <= luas <= area_maks:
            hasil[label == nomor] = 255

    return hasil


def hitung_cakupan_dan_jumlah(mask_sinyal, mask_kulit):
    area_kulit = max(1, int(np.count_nonzero(mask_kulit)))
    jumlah, _label, data_area, _ = cv.connectedComponentsWithStats(mask_sinyal, connectivity=8)
    daftar_area = [int(data_area[nomor, cv.CC_STAT_AREA]) for nomor in range(1, jumlah)]

    return {
        "cakupan_area": round((sum(daftar_area) / area_kulit) * 100, 2),
        "jumlah_titik": sum(1 for luas in daftar_area if luas >= 4),
        "mask": mask_sinyal,
    }


def nilai_tengah(data, area_valid):
    if np.any(area_valid):
        return float(np.median(data[area_valid]))
    return float(np.median(data))


def hitung_tekstur_glcm(abu_abu):
    contoh = cv.resize(abu_abu, (128, 128), interpolation=cv.INTER_AREA)
    contoh = (contoh / 4).astype(np.uint8)
    glcm = graycomatrix(contoh, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)

    ukuran = {
        "glcm_kontras": "contrast",
        "glcm_energi": "energy",
        "glcm_homogenitas": "homogeneity",
        "glcm_korelasi": "correlation",
    }
    return {nama: round(float(graycoprops(glcm, kode)[0, 0]), 4) for nama, kode in ukuran.items()}


def analisis_zona_wajah(zona_wajah, mask_kulit):
    biru, hijau, merah = cv.split(zona_wajah)
    abu_abu = cv.cvtColor(zona_wajah, cv.COLOR_BGR2GRAY)
    area_kulit = mask_kulit > 0

    dasar_cahaya = nilai_tengah(abu_abu, area_kulit)
    tingkat_merah = merah.astype(np.int16) - ((hijau.astype(np.int16) + biru.astype(np.int16)) // 2)
    dasar_merah = nilai_tengah(tingkat_merah, area_kulit)

    merah_kuat = tingkat_merah > max(30, dasar_merah + 24)
    merah_ringan = tingkat_merah > max(24, dasar_merah + 18)

    jerawat = jadi_mask(area_kulit & merah_kuat & (abu_abu < dasar_cahaya + 28))
    jerawat = saring_area_sinyal(jerawat, area_min=8, area_maks=160)

    noda_gelap = jadi_mask(area_kulit & (abu_abu < dasar_cahaya - 26) & ~merah_kuat)
    noda_gelap = saring_area_sinyal(noda_gelap, area_min=12, area_maks=380)

    kotak_5 = np.ones((5, 5), dtype=np.uint8)
    kotak_7 = np.ones((7, 7), dtype=np.uint8)
    kemerahan = jadi_mask(area_kulit & merah_ringan & (abu_abu > dasar_cahaya - 18))
    kemerahan = cv.morphologyEx(kemerahan, cv.MORPH_OPEN, kotak_5)
    kemerahan = cv.morphologyEx(kemerahan, cv.MORPH_CLOSE, kotak_7)

    garis = cv.Canny(cv.GaussianBlur(abu_abu, (3, 3), 0), 45, 120)
    garis_tipis = cv.getStructuringElement(cv.MORPH_RECT, (5, 1))
    kerutan = jadi_mask(area_kulit & (garis > 0) & (abu_abu < dasar_cahaya + 24))
    kerutan = cv.morphologyEx(kerutan, cv.MORPH_OPEN, garis_tipis)

    tekstur = cv.convertScaleAbs(cv.Laplacian(abu_abu, cv.CV_16S, ksize=3))
    dasar_tekstur = float(np.percentile(tekstur[area_kulit], 68)) if np.any(area_kulit) else float(np.percentile(tekstur, 68))
    pori_besar = jadi_mask(area_kulit & (tekstur > max(28, dasar_tekstur + 8)) & (tingkat_merah < dasar_merah + 32))
    pori_besar = cv.morphologyEx(pori_besar, cv.MORPH_OPEN, np.ones((2, 2), dtype=np.uint8))

    return {
        "jerawat": hitung_cakupan_dan_jumlah(jerawat, mask_kulit),
        "noda_gelap": hitung_cakupan_dan_jumlah(noda_gelap, mask_kulit),
        "kerutan": hitung_cakupan_dan_jumlah(kerutan, mask_kulit),
        "kemerahan": hitung_cakupan_dan_jumlah(kemerahan, mask_kulit),
        "pori_besar": hitung_cakupan_dan_jumlah(pori_besar, mask_kulit),
        "tekstur_glcm": hitung_tekstur_glcm(abu_abu),
    }


## 5. Fungsi Analisis Lengkap dan Overlay


In [ ]:
def hitung_skor(penalti_skor):
    return int(max(0, min(100, round(100 - penalti_skor))))


def tulis_label(gambar, teks, titik_awal, warna, ukuran_teks=0.45):
    x, y = titik_awal
    huruf = cv.FONT_HERSHEY_SIMPLEX
    (lebar_teks, tinggi_teks), garis_bawah = cv.getTextSize(teks, huruf, ukuran_teks, 1)

    x = max(0, min(x, gambar.shape[1] - lebar_teks - 6))
    y = max(tinggi_teks + 4, min(y, gambar.shape[0] - garis_bawah - 4))

    cv.rectangle(gambar, (x, y - tinggi_teks - 5), (x + lebar_teks + 6, y + garis_bawah + 3), warna, -1)
    cv.putText(gambar, teks, (x + 3, y), huruf, ukuran_teks, (255, 255, 255), 1, cv.LINE_AA)


def buat_overlay_hasil(wajah, kumpulan_mask):
    hasil = wajah.copy()
    lapisan_warna = np.zeros_like(hasil)

    for kode, mask in kumpulan_mask.items():
        lapisan_warna[mask > 0] = INFO_SINYAL[kode]["warna"]

    campuran = cv.addWeighted(hasil, 0.86, lapisan_warna, 0.30, 0)
    area_kena = np.any(lapisan_warna > 0, axis=2)
    hasil[area_kena] = campuran[area_kena]

    tinggi, lebar = hasil.shape[:2]
    warna_batas = (180, 166, 0)
    cv.rectangle(hasil, (2, 2), (lebar - 3, tinggi - 3), warna_batas, 2)
    tulis_label(hasil, "Area wajah", (10, 24), warna_batas)

    for _kode, nama_zona, (x, y, w, h) in bagi_zona_wajah(lebar, tinggi):
        cv.rectangle(hasil, (x, y), (x + w, y + h), warna_batas, 1)
        tulis_label(hasil, nama_zona, (x + 4, y + 16), warna_batas, ukuran_teks=0.38)

    for kode, mask in kumpulan_mask.items():
        warna = INFO_SINYAL[kode]["warna"]
        nama_pendek = INFO_SINYAL[kode]["singkat"]
        daftar_garis, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

        for urutan, garis_luar in enumerate(sorted(daftar_garis, key=cv.contourArea, reverse=True)[:8]):
            if cv.contourArea(garis_luar) < 5:
                continue
            x, y, w, h = cv.boundingRect(garis_luar)
            cv.rectangle(hasil, (x, y), (x + w, y + h), warna, 2)
            if urutan < 2:
                tulis_label(hasil, nama_pendek, (x, max(14, y - 4)), warna, ukuran_teks=0.38)

    return hasil


def analisis_foto(nama_berkas, gambar):
    wajah, _area_awal, _area_potong, wajah_ditemukan = siapkan_wajah(gambar)
    mask_kulit = buat_mask_kulit(wajah)
    tinggi, lebar = wajah.shape[:2]
    daftar_zona = bagi_zona_wajah(lebar, tinggi)

    jumlah_zona = len(daftar_zona)
    penjumlahan = {kode: {"cakupan_area": 0.0, "jumlah_titik": 0} for kode in INFO_SINYAL}
    kumpulan_mask = {kode: np.zeros((tinggi, lebar), dtype=np.uint8) for kode in INFO_SINYAL}
    baris_zona = []
    baris_tekstur = []

    for _kode_zona, nama_zona, (x, y, w, h) in daftar_zona:
        potongan_zona = wajah[y:y + h, x:x + w]
        kulit_zona = mask_kulit[y:y + h, x:x + w]
        hasil_zona = analisis_zona_wajah(potongan_zona, kulit_zona)

        cakupan_per_sinyal = {}
        for kode_sinyal in INFO_SINYAL:
            hasil_sinyal = hasil_zona[kode_sinyal]
            penjumlahan[kode_sinyal]["cakupan_area"] += float(hasil_sinyal["cakupan_area"])
            penjumlahan[kode_sinyal]["jumlah_titik"] += int(hasil_sinyal["jumlah_titik"])
            cakupan_per_sinyal[kode_sinyal] = float(hasil_sinyal["cakupan_area"])
            kumpulan_mask[kode_sinyal][y:y + h, x:x + w][hasil_sinyal["mask"] > 0] = 255

        penalti_zona = min(65.0, sum(cakupan_per_sinyal.values()) * 1.8)
        sinyal_terbesar = max(cakupan_per_sinyal, key=cakupan_per_sinyal.get)
        baris_zona.append({
            "nama_berkas": nama_berkas,
            "zona_wajah": nama_zona,
            "skor_zona": hitung_skor(penalti_zona),
            "sinyal_terbesar": INFO_SINYAL[sinyal_terbesar]["label"],
            "piksel_kulit": int(np.count_nonzero(kulit_zona)),
        })

        baris_tekstur.append({"nama_berkas": nama_berkas, "zona_wajah": nama_zona, **hasil_zona["tekstur_glcm"]})

    baris_fitur = []
    for kode_sinyal, info in INFO_SINYAL.items():
        rata_cakupan = penjumlahan[kode_sinyal]["cakupan_area"] / jumlah_zona
        jumlah_titik = int(penjumlahan[kode_sinyal]["jumlah_titik"])
        penalti = min(82.0, rata_cakupan * info["bobot_luas"] + min(24.0, jumlah_titik * info["bobot_jumlah"]))
        baris_fitur.append({
            "nama_berkas": nama_berkas,
            "fitur_sinyal": info["label"],
            "skor": hitung_skor(penalti),
            "cakupan_area": round(rata_cakupan, 2),
            "jumlah_titik": jumlah_titik,
        })

    overlay = buat_overlay_hasil(wajah, kumpulan_mask)
    skor_total = round(np.mean([baris["skor"] for baris in baris_fitur]))
    ringkasan = {
        "nama_berkas": nama_berkas,
        "wajah_terdeteksi": wajah_ditemukan,
        "skor_kesehatan_kulit": skor_total,
    }
    return wajah, overlay, baris_fitur, baris_zona, baris_tekstur, ringkasan


## 6. Unggah Foto dan Jalankan Analisis

Unggah maksimal 5 foto. Jika lebih dari 5, notebook hanya memproses 5 foto pertama agar tabel tetap rapi untuk laporan.


In [ ]:
unggahan = files.upload()
berkas_dipilih = list(unggahan.items())[:5]

if len(unggahan) > 5:
    print("Catatan: hanya 5 foto pertama yang diproses.")

if not berkas_dipilih:
    raise ValueError("Upload minimal 1 gambar.")

semua_baris_fitur = []
semua_baris_zona = []
semua_baris_tekstur = []
semua_ringkasan = []


def tampilkan_hasil(nama_berkas, gambar_awal, wajah, overlay, ringkasan):
    gambar_tampil = [
        ("Foto awal", gambar_awal),
        ("Wajah siap dibaca", wajah),
        ("Overlay hasil", overlay),
    ]

    fig, daftar_area = plt.subplots(1, 3, figsize=(12, 4))
    for area, (judul, gambar) in zip(daftar_area, gambar_tampil):
        area.imshow(cv.cvtColor(gambar, cv.COLOR_BGR2RGB))
        area.set_title(judul, fontsize=10)
        area.axis("off")

    status_wajah = "YA" if ringkasan["wajah_terdeteksi"] else "TIDAK"
    judul = f"{nama_berkas} | Skor: {ringkasan['skor_kesehatan_kulit']}/100 | Wajah terdeteksi: {status_wajah}"
    fig.suptitle(judul, fontsize=11)
    plt.tight_layout()
    plt.show()


for nama_berkas, isi_berkas in berkas_dipilih:
    gambar = cv.imdecode(np.frombuffer(isi_berkas, np.uint8), cv.IMREAD_COLOR)
    if gambar is None:
        print(f"File gagal dibaca: {nama_berkas}")
        continue

    wajah, overlay, baris_fitur, baris_zona, baris_tekstur, ringkasan = analisis_foto(nama_berkas, gambar)
    semua_baris_fitur.extend(baris_fitur)
    semua_baris_zona.extend(baris_zona)
    semua_baris_tekstur.extend(baris_tekstur)
    semua_ringkasan.append(ringkasan)

    tampilkan_hasil(nama_berkas, gambar, wajah, overlay, ringkasan)

print(f"Jumlah foto diproses: {len(semua_ringkasan)}")


## 7. Tabel Data Hasil Analisis

Tabel dibuat dalam format ringkas untuk laporan. Jika 5 foto diunggah, semua data 5 foto akan tampil.


In [ ]:
tabel_ringkasan = pd.DataFrame(semua_ringkasan)
tabel_fitur = pd.DataFrame(semua_baris_fitur)
tabel_zona = pd.DataFrame(semua_baris_zona)
tabel_tekstur = pd.DataFrame(semua_baris_tekstur)

if tabel_ringkasan.empty or tabel_fitur.empty:
    raise ValueError("Belum ada data pemeriksaan. Jalankan cell upload terlebih dahulu.")


def buat_tabel_ringkas(kolom_nilai):
    return (
        tabel_fitur
        .pivot_table(index="nama_berkas", columns="fitur_sinyal", values=kolom_nilai, aggfunc="first")
        .reindex(columns=URUTAN_SINYAL)
        .reset_index()
        .rename(columns={"nama_berkas": "Nama Berkas"})
    )


tabel_ringkasan_rapi = tabel_ringkasan.rename(columns={
    "nama_berkas": "Nama Berkas",
    "wajah_terdeteksi": "Wajah Terdeteksi",
    "skor_kesehatan_kulit": "Skor Kesehatan Kulit",
})
tabel_ringkasan_rapi["Wajah Terdeteksi"] = tabel_ringkasan_rapi["Wajah Terdeteksi"].map({True: "YA", False: "TIDAK"})

tabel_skor = buat_tabel_ringkas("skor")
tabel_cakupan = buat_tabel_ringkas("cakupan_area").round(2)
tabel_jumlah = buat_tabel_ringkas("jumlah_titik").fillna(0).astype({nama: int for nama in URUTAN_SINYAL})

tabel_zona_rapi = tabel_zona.rename(columns={
    "nama_berkas": "Nama Berkas",
    "zona_wajah": "Zona Wajah",
    "skor_zona": "Skor Zona",
    "sinyal_terbesar": "Sinyal Terbesar",
    "piksel_kulit": "Piksel Kulit",
}).reset_index(drop=True)

tabel_tekstur_rapi = tabel_tekstur.rename(columns={
    "nama_berkas": "Nama Berkas",
    "zona_wajah": "Zona Wajah",
    "glcm_kontras": "Kontras",
    "glcm_energi": "Energi",
    "glcm_homogenitas": "Homogenitas",
    "glcm_korelasi": "Korelasi",
}).round(4).reset_index(drop=True)

tabel_sebaran = (
    tabel_fitur
    .groupby("fitur_sinyal")[["skor", "cakupan_area", "jumlah_titik"]]
    .std(ddof=0)
    .reset_index()
    .rename(columns={
        "fitur_sinyal": "Fitur Sinyal",
        "skor": "Sebaran Skor",
        "cakupan_area": "Sebaran Cakupan",
        "jumlah_titik": "Sebaran Jumlah",
    })
    .round(4)
)

daftar_tabel = [
    ("A. Ringkasan Hasil per Foto", tabel_ringkasan_rapi),
    ("B. Data Skor per Sinyal Kulit", tabel_skor),
    ("C. Data Cakupan Area (%)", tabel_cakupan),
    ("D. Data Jumlah Titik", tabel_jumlah),
    ("E. Data Zona Wajah", tabel_zona_rapi),
    ("F. Data Tekstur GLCM per Area", tabel_tekstur_rapi),
    ("G. Hasil Standar Deviasi", tabel_sebaran),
]

for judul, tabel in daftar_tabel:
    print(judul)
    display(tabel)


## Catatan Metode

Pipeline yang dipakai di notebook ini:

1. **Input image**: user upload foto wajah sebagai data awal.
2. **Face detection**: OpenCV Haar Cascade dipakai untuk mencari bounding box wajah.
3. **ROI cropping**: area wajah dipotong supaya background tidak ikut masuk hitungan.
4. **CLAHE normalization**: kontras wajah distabilkan agar foto dengan cahaya berbeda tetap lebih mudah dibaca.
5. **Color space conversion**: gambar dibaca dalam BGR, grayscale, HSV, YCrCb, dan LAB sesuai kebutuhan fitur.
6. **Skin mask**: area kulit dipisahkan dari area non-kulit.
7. **Face zoning**: wajah dibagi menjadi dahi, pipi kiri, pipi kanan, hidung, dan dagu.
8. **Adaptive thresholding**: nilai piksel dibandingkan dengan median zona, bukan satu threshold global untuk semua foto.
9. **Morphology**: opening dan closing dipakai untuk membersihkan noise pada mask.
10. **Connected components**: area sinyal yang saling terhubung dihitung sebagai satu blob.
11. **Canny edge detection**: edge halus dipakai sebagai salah satu sinyal kerutan.
12. **Laplacian texture response**: respon tekstur lokal dipakai untuk membaca sinyal pori.
13. **GLCM texture features**: contrast, energy, homogeneity, dan correlation dihitung sebagai fitur tekstur.
14. **Visual overlay**: mask sinyal diberi warna di atas area wajah.
15. **Summary table**: hasil mask diringkas menjadi coverage, count, score, dan standar deviasi.

Batasannya: hasil ini adalah pemetaan sinyal visual pada foto, bukan diagnosis medis.
